# Initial design thoughts


## Given

- resin
    - One `Resin` object
        - `ResinConstituentAbsorber` objects
            - `AbsorberSpectrum` objects with w/w concentrations for absorbers and photoinitiators
        - `ResinConstituentMonomer` object(s)
            - `Monomer` object(s) with %monomer concentration(s)
- source
    - One `SourceSpectrum` object
    

## Calculate

- D_n(z)
    - Array
    - Plot
- Model 1: h_a
- Model 2: a, b


# Code

## Imports

In [1]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget

from scipy.optimize import curve_fit
import scipy.integrate as integrate

import spectra.data as data
from spectra.resin import ResinConstituentAbsorber, ResinConstituentMonomer, Resin

## Models

In [2]:
def ha_model(z, b):
    return np.exp(-z / b)


def a_b_model(z, a, b):
    return a * np.exp(-z / b) + (1 - a)


def a_b_c_model(z, a, b, c):
    return a * np.exp(-z / b) + c


def fit_ha_model(z, dprime):
    param_bounds = ([0.1], [np.inf])
    popt, pcov = curve_fit(ha_model, z, dprime, bounds=param_bounds)
    return popt, pcov


def fit_a_b_model(z, dprime):
    param_bounds = ([0., 0.1], [1., np.inf])
    popt, pcov = curve_fit(a_b_model, z, dprime, bounds=param_bounds)
    return popt, pcov


def fit_a_b_c_model(z, dprime):
    param_bounds = ([0., 0.1, 0.], [1., np.inf, 1.])
    popt, pcov = curve_fit(a_b_c_model, z, dprime, bounds=param_bounds)
    return popt, pcov


## ResinSourceCombo

In [3]:
class ResinSourceCombo:
    
    def __init__(self, *, source, resin, dn_calc_config):
        
        # Store inputs
        self.source = source
        self.resin = resin
        self.dn_calc_config = dn_calc_config
        
        # Pre-calculate source integral over wavelengths
        self.source_integral = integrate.simps(self.source.power, self.source.wavelength)
        
        # Calculate normalized dose as a function of z
        self.z_um = self._calc_z()
        self.normalized_dose = self._calc_normalized_dose()
        
        # Fit normalized dose to models
        self.ha_model = self._calc_ha_model()
        self.a_b_model = self._calc_a_b_model()
        self.a_b_c_model = self._calc_a_b_c_model()
        
    def _calc_z(self):
        if self.dn_calc_config["sampling"] == 'linear':
            z = np.linspace(0, self.dn_calc_config["z_max_um"], self.dn_calc_config["num_pnts"])
        else:
            z = np.logspace(-1, 5, num=self.dn_calc_config["num_pnts"])
        return z

    def _calc_normalized_dose(self):
        
        # Set up wavelengths array
        wavelengths = self.source.wavelength
        
        # Set up source power array
        power = self.source.power
        
        # Set up absorption coefficient array for wavelengths
        absorption_coeff_inverse_um = 1e-4 * resin.calc_absorption_coef_inv_cm(wavelengths)
        
        # Calculate normalized dose as a function of z
        norm_dose = np.zeros(len(self.z_um))
        for i, z_value in enumerate(self.z_um):
            norm_dose[i] = integrate.simps(power * \
                np.exp(-absorption_coeff_inverse_um * z_value), wavelengths) / self.source_integral
        return norm_dose

    def _calc_ha_model(self):
        popt, pcov = fit_ha_model(self.z_um, self.normalized_dose)
        ha_model = {}
        ha_model['ha'] = popt[0]
        ha_model['stdev'] = np.sqrt(np.diag(pcov))
        return ha_model

    def _calc_a_b_model(self):
        popt, pcov = fit_a_b_model(self.z_um, self.normalized_dose)
        a_b_model = {}
        a_b_model['a'] = popt[0]
        a_b_model['b'] = popt[1]
        a_b_model['stdev'] = np.sqrt(np.diag(pcov))
        return a_b_model

    def _calc_a_b_c_model(self):
        popt, pcov = fit_a_b_c_model(self.z_um, self.normalized_dose)
        a_b_c_model = {}
        a_b_c_model['a'] = popt[0]
        a_b_c_model['b'] = popt[1]
        a_b_c_model['c'] = popt[2]
        a_b_c_model['stdev'] = np.sqrt(np.diag(pcov))
        return a_b_c_model

    def plot(self, title='', file=''):
        fig, ax = plt.subplots()
        ax.plot(self.z_um, self.normalized_dose, 'r', label="$D_n(z)$")
        
        text_func3 = 'fit: y = a*exp(-z/b) + c\n'
        model = self.a_b_c_model
        ax.plot(self.z_um, a_b_c_model(self.z_um, model['a'], model['b'], model['c']), 'r--', 
                label=text_func3 + 'a={:4.3}, b={:5.3f}, c={:5.3f}'.format(model['a'], model['b'], model['c']))
        
        text_func2 = 'fit: y = a*exp(-z/b) + (1-a)\n'
        model = self.a_b_model
        ax.plot(self.z_um, a_b_model(self.z_um, model['a'], model['b']), 'g-.', 
                label=text_func2 + 'a={:4.3}, b={:5.3f}'.format(model['a'], model['b']))
        
        text_func1 = 'fit: y = exp(-z/b)\n'
        model = self.ha_model
        ax.plot(self.z_um, ha_model(self.z_um, model['ha']), 'r:', 
                label=text_func1 + 'b={:7.5f}'.format(model['ha']))

        ax.set_xlim(0, )
        ax.set_xlabel('z ($\mu$m)')
        ax.set_ylabel("D'(z)")
        ax.set_ylim(0,1)
        ax.legend()
        
        if title:
            ax.set_title(title)
        
        if file:
            plt.savefig(file)
            
        return fig, ax        

In [4]:
dn_calc_config = {
    "z_max_um": 100,
    "num_pnts": 201,
    "sampling": "linear",
}

In [5]:
# Create resin
pegda = ResinConstituentMonomer(data.monomers['PEGDA'], 100)
avobenzone = ResinConstituentAbsorber(
    data.absorbers['avobenzone_in_PEGDA_2017-04-13'],
    0.38
)
irgacure819 = ResinConstituentAbsorber(
    data.photoinitiators['irgacure819_in_PEGDA_2017-04-13'],
    1.0
)
resin = Resin(monomers=pegda, absorbers=avobenzone, photoinitiators=irgacure819)

# Source
source = data.sources['HR3.2_365nm_370nm_short_pass_filter-2019-12-31']


temp_combo = ResinSourceCombo(source=source, resin=resin, dn_calc_config=dn_calc_config)

print(temp_combo.source.name)

temp_combo.plot();

HR3.2_365nm_370nm_short_pass_filter-2019-12-31


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [6]:
vars(temp_combo)

{'source': <spectra.sourcespectrum.SourceSpectrum at 0x7fef51def690>,
 'resin': <spectra.resin.Resin at 0x7fef51e7e7d0>,
 'dn_calc_config': {'z_max_um': 100, 'num_pnts': 201, 'sampling': 'linear'},
 'source_integral': 10.91797235674928,
 'z_um': array([  0. ,   0.5,   1. ,   1.5,   2. ,   2.5,   3. ,   3.5,   4. ,
          4.5,   5. ,   5.5,   6. ,   6.5,   7. ,   7.5,   8. ,   8.5,
          9. ,   9.5,  10. ,  10.5,  11. ,  11.5,  12. ,  12.5,  13. ,
         13.5,  14. ,  14.5,  15. ,  15.5,  16. ,  16.5,  17. ,  17.5,
         18. ,  18.5,  19. ,  19.5,  20. ,  20.5,  21. ,  21.5,  22. ,
         22.5,  23. ,  23.5,  24. ,  24.5,  25. ,  25.5,  26. ,  26.5,
         27. ,  27.5,  28. ,  28.5,  29. ,  29.5,  30. ,  30.5,  31. ,
         31.5,  32. ,  32.5,  33. ,  33.5,  34. ,  34.5,  35. ,  35.5,
         36. ,  36.5,  37. ,  37.5,  38. ,  38.5,  39. ,  39.5,  40. ,
         40.5,  41. ,  41.5,  42. ,  42.5,  43. ,  43.5,  44. ,  44.5,
         45. ,  45.5,  46. ,  46.5,  47. ,  4

In [7]:
data.sources

{'365nm_visitech_with_50mm_asahi_filter_2018-11-22': <spectra.sourcespectrum.SourceSpectrum at 0x7fef51e147d0>,
 'Asiga_405nm_100um_fiber-2016-12-14': <spectra.sourcespectrum.SourceSpectrum at 0x7fef51e14cd0>,
 'HR1_385nm_100um_fiber-2016-12-14': <spectra.sourcespectrum.SourceSpectrum at 0x7fef51e14b90>,
 'HR2_385nm_2020-03-23': <spectra.sourcespectrum.SourceSpectrum at 0x7fef51e14950>,
 'HR3.2_365nm_370nm_short_pass_filter-2019-12-31': <spectra.sourcespectrum.SourceSpectrum at 0x7fef51def690>,
 'HR3.2_365nm_no_filter-2019-12-31': <spectra.sourcespectrum.SourceSpectrum at 0x7fef51dfe050>,
 'HR3.3_365nm_370nm_short_pass_filter-2020-03-23': <spectra.sourcespectrum.SourceSpectrum at 0x7fef51e01050>,
 'HR3.3_365nm_no_filter-2020-03-23': <spectra.sourcespectrum.SourceSpectrum at 0x7fef516b59d0>,
 'HR3.3_365nm_through_tray_370nm_short_pass_filter-2020-03-23': <spectra.sourcespectrum.SourceSpectrum at 0x7fef51df5f90>,
 'HR3.3_365nm_through_tray_no_filter-2020-03-23': <spectra.sourcespectrum.S

In [19]:
vars(temp_combo.source)

{'name': 'HR3.2_365nm_370nm_short_pass_filter-2019-12-31',
 '_data': array([[2.98026000e+02, 0.00000000e+00],
        [2.98106000e+02, 0.00000000e+00],
        [2.98185000e+02, 0.00000000e+00],
        ...,
        [4.34114000e+02, 6.98319005e-03],
        [4.34167000e+02, 6.98319005e-03],
        [4.34221000e+02, 6.98319005e-03]]),
 'wavelength': array([298.026, 298.106, 298.185, ..., 434.114, 434.167, 434.221]),
 'power': array([0.        , 0.        , 0.        , ..., 0.00698319, 0.00698319,
        0.00698319])}

In [20]:
vars(temp_combo.resin)

{'monomers': [ResinConstituentMonomer(material=Monomer(/Users/nordin/Documents/Projects/3D_printer_spectra_repos/3dprinter_spectra_and_tools/spectra/data/monomers/pegda.json), concentration_percent_monomer=100)],
 'absorbers': [ResinConstituentAbsorber(material=<spectra.absorberspectrum.AbsorberSpectrum object at 0x7f97d0418250>, concentration_ww_percent=0.38)],
 'photoinitiators': [ResinConstituentAbsorber(material=<spectra.absorberspectrum.AbsorberSpectrum object at 0x7f97d041b090>, concentration_ww_percent=1.0)],
 '_absorbers': [ResinConstituentAbsorber(material=<spectra.absorberspectrum.AbsorberSpectrum object at 0x7f97d0418250>, concentration_ww_percent=0.38),
  ResinConstituentAbsorber(material=<spectra.absorberspectrum.AbsorberSpectrum object at 0x7f97d041b090>, concentration_ww_percent=1.0)],
 'name': 'PEG-100__Avo-0.38__Irg-1',
 '_density': 1110.0,
 '_molar_concentrations': [0.013589355327169046, 0.026523297491039426],
 'wavelength': array([299.969, 300.767, 301.565, 302.363, 

In [21]:
print(temp_combo.resin.name)
print()

for a in temp_combo.resin.monomers:
    print(f"{a.material.name} - {a.concentration_percent_monomer}%")
    print()

for a in temp_combo.resin.absorbers:
    print(f"{a.material.name} - {a.concentration_ww_percent}%")
    print()
    
for a in temp_combo.resin.photoinitiators:
    print(f"{a.material.name} - {a.concentration_ww_percent}%")
    print()

PEG-100__Avo-0.38__Irg-1

PEGDA - 100%

Avobenzone - 0.38%

Irgacure 819 - 1.0%

